# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# Ollama CLI에서 LangChain API로

**목표:** LangChain 공식 API의 입력·반환값과 실행 위치를 직접 설명합니다.

[교재 안내](../../README.md) · [API 상세 해설](../../LANGCHAIN_API.md) · [슬라이드 6~22쪽](../../../슬라이드/강의자료_2일차_v2.pdf#page=6)

**커널:** Python (Day2 LangChain). 설치·커널 선택을 완료한 뒤 코드 셀을 순서대로 실행하세요. 본문은 LangChain API를 직접 사용합니다. 모의 예제와 실제 요청을 구분하며 정답 출력은 미리 넣지 않습니다.

60분 동안 API 설명과 실행 확인을 번갈아 진행합니다. 상세 옵션·추가 비교는 복습 자료로도 사용합니다. 마지막 시간에는 TODO를 진행합니다.

## 1. Jetson 호스트에서 Ollama CLI 체험

호스트 터미널에서 아래 명령으로 Ollama 컨테이너에 들어갑니다. `ollama list`에 gemma3:1b가 있는지 확인하고 같은 질문을 직접 입력합니다. 수업 모델을 다시 다운로드하지 않습니다.

```bash
docker exec -it physical_ai_ollama /bin/sh
ollama list
ollama run gemma3:1b
```

`센서의 역할을 한 문장으로 설명하세요.`를 입력해 응답을 봅니다. `/bye`는 모델 대화 종료, `exit`는 컨테이너 셸 종료입니다. Python·pip 명령은 여기서 실행하지 않습니다. CLI의 대화 이력은 다음 Jupyter 요청으로 전달되지 않습니다.

## 2. 개인 컨테이너에서 설치·커널 선택

2일차 업데이트 파일을 본인의 자료 폴더에 덮어쓴 뒤 컨테이너를 엽니다. 로컬·원격 사용자 모두 개인 컨테이너의 **Jupyter Terminal**에서 아래 명령을 실행합니다. 업데이트 적용 때마다 이 준비 절차를 다시 확인합니다. 설치 스크립트는 Data/venvs에 환경을 만들고 이 설치에만 PIP_NO_INDEX를 해제합니다.

```bash
cd /workspace
bash 04.LLM_v2/scripts/install_langchain.sh
PY=Data/venvs/day2-langchain/bin/python
"$PY" 04.LLM_v2/scripts/check_langchain.py \
  --server
```

설치와 `Ollama 모델·vision PASS`를 확인한 후 노트북 커널을 **Python (Day2 LangChain)**으로 선택합니다. 설치 중에는 아래 설명만 읽고 import 셀은 완료 후 실행하세요.

In [ ]:
from pathlib import Path
import json, os, sys, time
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / '04.LLM_v2').is_dir()), None)
if ROOT is None:
    raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE = ROOT / '04.LLM_v2'
sys.path.insert(0, str(COURSE))
print('Python:', sys.executable)
print('교재:', COURSE)
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda

## 3. 실행 위치와 요청 경로

학생 Python과 LangChain은 개인 컨테이너에서 실행됩니다. ChatOllama가 HTTP로 요청하면 별도 Ollama 컨테이너의 Gemma가 계산합니다. 응답은 Ollama API를 거쳐 LangChain이 AIMessage로 돌려줍니다.

![실행 위치와 요청·응답](../../assets/01_ollama_basics-02.svg)

### LangChain API · 패키지와 import 경로를 구분합니다

LangChain은 Python에서 모델 입력과 처리 단계를 연결하는 라이브러리입니다. 호스트의 `ollama run gemma3:1b`는 CLI 대화이고, 아래 import는 개인 컨테이너의 Python 코드입니다. CLI에서 나눈 대화는 Jupyter로 전달되지 않습니다.

| 설치 패키지 | import 경로 | 이번 수업에서 쓰는 객체 |
|---|---|---|
| langchain-ollama | langchain_ollama | ChatOllama |
| langchain-core | langchain_core.messages | SystemMessage, HumanMessage, AIMessage |
| langchain-core | langchain_core.prompts | ChatPromptTemplate, MessagesPlaceholder |
| langchain-core | langchain_core.runnables | RunnableLambda |
| langchain-core | langchain_core.documents | Document |
| langchain-text-splitters | langchain_text_splitters | RecursiveCharacterTextSplitter |

하이픈은 설치 패키지 이름에, 밑줄은 Python 모듈 이름에 사용합니다. 수업은 requirements-langchain.txt의 고정 버전을 사용합니다. import는 클래스를 가져오는 단계이며 모델 답변을 생성하지 않습니다. 설치가 끝나고 **Python (Day2 LangChain)** 커널을 선택한 다음 실행하세요.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage,
)
print(ChatOllama.__name__)
print(HumanMessage.__name__)

### LangChain API · SystemMessage·HumanMessage·AIMessage

메시지는 **역할과 내용**을 함께 보관하는 Python 객체입니다. `SystemMessage(content=...)`는 공통 지시, `HumanMessage(content=...)`는 사용자 입력, `AIMessage(content=...)`는 모델 응답을 표현합니다. 아래 AIMessage는 구조 관찰을 위해 직접 만든 예시이며 실제 모델 응답이 아닙니다.

`content`는 이 텍스트 실습에서는 문자열입니다. 다른 모델이나 멀티모달 요청에서는 콘텐츠 블록 목록일 수도 있으므로 모든 상황에서 문자열이라고 단정하지 않습니다. `message.type`은 각각 system/human/ai입니다. Ollama 전송 시 역할 변환은 통합 라이브러리가 처리합니다.

입력은 순서가 있는 메시지 목록입니다. system과 human은 서로 변환되는 단계가 아닙니다. system 지시가 있더라도 사실 정확성이 보장되는 것은 아닙니다.

In [ ]:
messages = [
    SystemMessage(content="한국어로 짧게 답하세요."),
    HumanMessage(content="센서란 무엇인가요?"),
]
for message in messages:
    print(message.type, message.content)
example_reply = AIMessage(content="구조 확인용 예시입니다.")
print(example_reply.type)

![메시지와 응답 속성](../../assets/01_ollama_basics-05.svg)

### LangChain API · ChatOllama 생성자의 필수 설정

`ChatOllama(...)`의 반환값은 모델 호출에 사용할 **ChatOllama 객체**입니다. 서버 자체나 모델 가중치를 Python 안에 생성하는 것이 아닙니다. 이 수업 설정에서는 생성자만 실행해 답변을 요청하지 않습니다.

| 인자 | 의미 | 수업 설정 |
|---|---|---|
| model | Ollama 서버에 준비된 모델 태그 | gemma3:1b |
| base_url | Ollama HTTP API 주소 | http://127.0.0.1:11434 |
| temperature | 출력 샘플링의 무작위성 조절 | 0 |
| client_kwargs | HTTP 클라이언트 설정 사전 | timeout: 180초 |

127.0.0.1은 **코드가 실행되는 곳**을 가리킵니다. 이번 배포는 Jetson의 개인 컨테이너와 Ollama 컨테이너가 host network를 사용합니다. 학생 PC에서 별도 Python을 실행하는 상황과 혼동하지 마세요. 아래에서 model과 base_url을 출력해 어떤 서버의 어떤 모델을 사용할지 먼저 확인합니다.

In [ ]:
model = ChatOllama(
    model="gemma3:1b",
    base_url="http://127.0.0.1:11434",
    temperature=0,
    seed=2026,
    num_ctx=4096,
    num_predict=256,
    keep_alive="5m",
    client_kwargs={"timeout": 180},
)
print(model.model, model.base_url)

### LangChain API · 생성 옵션과 실행 시간을 읽습니다

`num_ctx`는 문맥 창의 토큰 크기 설정이고 `num_predict`는 생성할 토큰 수의 상한입니다. 글자 수·문장 수·대화 쌍 수와 다릅니다. 입력 메시지와 출력에 필요한 공간을 함께 고려해야 합니다.

`temperature=0`과 `seed=2026`은 비교 조건을 고정하기 위한 설정입니다. 답변의 사실 정확성이나 모든 환경에서의 완전한 재현성을 보장하지 않습니다. `keep_alive="5m"`는 요청 이후 모델을 메모리에 유지할 시간을 지정하며 대화 기억 저장 기간이 아닙니다. `client_kwargs={"timeout":180}`은 HTTP 클라이언트의 초 단위 타임아웃 설정이며 생성 토큰 상한과 다른 제약입니다.

실습에서는 먼저 기본 설정을 유지합니다. 출력이 잘리면 답변 원문과 done_reason을 확인한 후 상한이나 질문 길이를 조정합니다. 여러 옵션을 동시에 바꾸면 원인을 비교하기 어렵습니다.

In [ ]:
settings = {
    "temperature": model.temperature,
    "seed": model.seed,
    "num_ctx": model.num_ctx,
    "num_predict": model.num_predict,
    "keep_alive": model.keep_alive,
}
print(settings)

## 4. 첫 실제 모델 요청

아래 셀에서 처음으로 Jupyter에서 모델에 요청합니다. 생성자가 아니라 invoke가 요청을 실행합니다. 첫 요청에는 모델 적재 시간이 포함될 수 있습니다. 네트워크·모델 오류가 나면 원문을 확인하고 다음 셀로 넘어가기 전에 해결하세요.

In [ ]:
messages = [HumanMessage(content='센서의 역할을 한 문장으로 설명하세요.')]
started = time.perf_counter()
reply = model.invoke(messages)
elapsed = time.perf_counter() - started
print(reply.content)
print(reply.response_metadata)
print('전체 경과 초:', elapsed)

### LangChain API · invoke의 입력과 반환값

`model.invoke(input)`은 입력을 보내고 완성된 응답을 기다리는 동기 호출입니다. 이 수업에서는 `list[BaseMessage]`를 전달하고 `AIMessage`를 받습니다. 문자열 한 개나 ChatPromptValue도 입력으로 사용할 수 있지만, 역할을 눈으로 확인하려고 메시지 목록을 먼저 사용합니다.

`model.invoke("센서란?")`는 사용자 입력만 주는 편의 방식입니다. system 규칙이 자동으로 추가되는 것은 아닙니다. 앞 CLI 대화나 이전 invoke 결과도 자동으로 다음 요청에 포함되지 않습니다.

`prompt.invoke(dict)`는 뒤에서 배울 프롬프트 구성만 실행하며, `model.invoke(messages)`와 실행 대상이 다릅니다. 이름이 invoke라고 해서 항상 모델을 호출하는 것은 아닙니다. 아래 셀은 앞 호출의 반환 타입만 확인하므로 추가 모델 요청이 없습니다.

In [ ]:
print(type(reply).__name__)
print(isinstance(reply, AIMessage))
print(reply.content)
# 실제 요청: reply = model.invoke(messages)
# 위 첫 호출 셀의 reply를 재사용합니다.

### LangChain API · 응답의 본문·메타데이터·토큰 정보

AIMessage에서 `content`, `response_metadata`, `usage_metadata`는 각각 읽는 속성입니다. content에서 metadata가 만들어지는 직렬 처리 과정이 아닙니다.

| 속성 | 이 수업에서 확인하는 내용 |
|---|---|
| content | 응답 본문. JSON 응답도 먼저 문자열로 도착합니다. |
| response_metadata | 모델 이름, 종료 이유, 실행 시간 등 서버 실행 정보 |
| usage_metadata | 제공되는 경우 입력·출력·전체 토큰 수 |
| model_dump(mode="json") | JSON 저장에 맞는 사전 표현. 문자열 JSON을 만들지는 않습니다. |

response_metadata의 키는 제공자·버전·응답에 따라 다를 수 있어 get을 사용합니다. Ollama의 total_duration과 load_duration은 나노초 단위이므로 초로 표시할 때 1e9로 나눕니다. total_duration은 서버 측 시간이며 Python에서 재는 전체 경과 시간과 다릅니다. done_reason이 length면 출력이 상한으로 잘렸는지 검토합니다. 메타데이터는 사실 정답 여부를 판정하지 않습니다.

In [ ]:
metadata = reply.response_metadata
print("종료 이유:", metadata.get("done_reason"))
for key in ("total_duration", "load_duration"):
    value = metadata.get(key)
    print(key, None if value is None else value / 1e9)
print("토큰:", reply.usage_metadata)
serialized = reply.model_dump(mode="json")
print(type(serialized).__name__)

### LangChain API · 응답 객체와 기록용 사전을 구분합니다

본문의 모델 호출은 `ChatOllama` 생성자와 `invoke`를 직접 사용합니다. `model.invoke(messages)`의 반환값은 `AIMessage`이며 답변 본문은 `reply.content`로 읽습니다.

응답을 저장할 때는 필요한 속성을 꺼내 Python 사전 `record`를 만듭니다. `record["answer"]`는 우리가 정한 사전 키이며 `AIMessage`의 속성이 아닙니다. 응답 객체와 기록용 사전의 타입·접근 방법을 구분하세요.

파일 저장, 대화 목록 변경, 타임아웃 처리, 출처 검사는 애플리케이션 코드의 책임입니다. LangChain을 import했다고 이 기능이 모두 자동으로 생기는 것은 아닙니다.


In [ ]:
# API 반환 객체와 저장용 사전을 구분합니다.
record = {
    "answer": reply.content,
    "metadata": reply.response_metadata,
    "usage": reply.usage_metadata,
}
print(type(reply).__name__, type(record).__name__)
print(record["answer"])

## 5. 공통 지시 → 역할 추가 → 같은 역할에 자료 추가

모델과 질문, 공통 지시는 유지합니다. 두 번째에는 역할을 추가하고 세 번째에는 **그 역할을 유지한 채 자료만 추가**합니다. 짧게 답하기·정보 부족 밝히기는 모든 입력에 공통입니다. 비교는 기본/역할/역할+자료의 세 조건입니다.

![프롬프트 비교 조건](../../assets/01_ollama_basics-07.svg)

가상 지침은 실제 제조 규격이 아닙니다. 질문을 실행하기 전에 지침에서 기대 근거를 찾으세요.

In [ ]:
BASE_RULE = '한국어로 간결하게 답하세요. 주어진 정보가 부족하면 부족하다고 밝히세요.'
STEP_RULE = ('조건을 먼저 확인하고 문제를 단계별로 검토하세요. '
             '출력은 결론과 확인 가능한 짧은 근거 2개 이내로 작성하세요.')

In [ ]:
GUIDELINE = (COURSE / 'data/work_instruction.txt').read_text().strip()
role_rule = BASE_RULE + ' 작업대 안내 도우미입니다.'
rules = [BASE_RULE, role_rule, role_rule + '\n다음 자료만 사용하세요.\n' + GUIDELINE]
print(GUIDELINE)

## 6. 세 입력을 LangChain으로 직접 비교

각 요청의 SystemMessage만 바꿉니다. invoke가 돌려준 AIMessage에서 본문과 메타데이터를 읽고 Python 사전에 기록합니다. 저장 파일이나 대화 기억은 자동으로 생성되지 않습니다. 아래 comparisons는 이 노트북에서 만든 관찰 목록입니다.

In [ ]:
question = '파란 블록이 가려지면 어떻게 하나요?'
comparisons = []
for label, rule in zip(['기본', '역할', '역할+자료'], rules):
    messages = [SystemMessage(rule), HumanMessage(question)]
    started = time.perf_counter()
    result = {'label': label, 'question': question,
              'messages': [m.model_dump(mode='json') for m in messages]}
    try:
        response = model.invoke(messages)
        result.update(answer=response.content, metadata=response.response_metadata,
                      usage=response.usage_metadata)
        if not response.content or response.response_metadata.get('done_reason') == 'length':
            raise ValueError('빈 응답 또는 출력 상한으로 잘린 응답입니다.')
        result['status'] = 'ok'
    except Exception as exc:
        result.update(status='request_error', error=f'{type(exc).__name__}: {exc}')
    result['seconds'] = time.perf_counter() - started
    comparisons.append(result)
    print(label, result['status'], result.get('answer'), result.get('error'), result['seconds'])

## 7. 자료 밖 질문과 출력 기록

자료가 없는 온도 질문은 답변을 추측하지 않는지 확인합니다. 응답 도착과 사실 정답을 구분하세요. 로그 저장은 Python 표준 라이브러리로 수행합니다. model_dump(mode="json")은 객체를 사전으로 바꾸고 json.dumps는 사전을 JSON 문자열로 바꿉니다.

In [ ]:
outside_question = '설비의 허용 최고 온도는 몇 도인가요?'
try:
    outside_reply = model.invoke([SystemMessage(rules[-1]), HumanMessage(outside_question)])
    print(outside_reply.content)
except Exception as exc:
    print(type(exc).__name__, str(exc))
from uuid import uuid4
output_dir = ROOT / 'Data/outputs/04.LLM_v2' / uuid4().hex
output_dir.mkdir(parents=True)
log_path = output_dir / 'prompt-comparison.json'
log_path.write_text(json.dumps(comparisons, ensure_ascii=False, indent=2))
print(log_path)

## 8. 다음 교시 연결: 기억과 단계별 지시

다음 교시는 MessagesPlaceholder와 프롬프트 파이프라인을 사용합니다. 단계별 검토 지시는 모델 입력 조건이며 모델의 실제 내부 사고 공개나 정답 보장이 아닙니다. 조건을 확인한 결론과 짧은 근거를 비교합니다.

## 마지막 TODO — 설치 확인과 프롬프트 관찰

**코드 작성: 없음 · 실행·관찰 실습 · 마지막 5~10분 안에 진행**

- [ ] Day2 LangChain 커널을 선택하고 sys.executable의 경로를 기록합니다.
- [ ] 제공된 프롬프트 세 가지를 실행하고 답변·오류·지연을 확인합니다.
- [ ] 자료에 있는 질문과 없는 질문을 하나씩 정합니다.
- [ ] 어느 입력 조건에서 답변이 달라졌는지 원문을 근거로 설명합니다.

함수를 새로 작성하는 TODO는 없습니다. 설치 명령 실행이나 경로 입력은 코드 작성 과제로 세지 않습니다. 설치가 늦은 학생은 필수 요청 한 건을 먼저 수행하고 나머지 비교를 마무리합니다.

| 확인 항목 | 내 기록 |
|---|---|
| 실제 커널 Python 경로 | 직접 작성 |
| 자료에 있는 질문 / 기대 근거 | 직접 작성 |
| 자료에 없는 질문 / 실제 응답 | 직접 작성 |
| 비교한 입력 조건과 답변 차이 | 직접 작성 |

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.